# M1.0c Primer: Gradient, Divergence, Curl and Laplacian

Book pointer: chapter 1, section 1.4 (differentiation).

Third of four primer notebooks. This is the heart of vector calculus: four operators that describe how a field changes from point to point. Each one is introduced as a measurement you could make with a tiny instrument, derived from that measurement, and only then written as a formula. The notebook is meant to be read on its own. Section 2 teaches the whole topic in prose; section 3 goes through it again slowly with hand-worked numbers and code that reproduces them; sections 6 and 7 collect the formulas and answer the warm-up questions.

## 1. Warm-up

Try these on paper before reading on. Answers are in section 7.

1. What is a partial derivative? How do you compute $\partial f/\partial y$ for $f = x^2 y + \sin x$?
2. Which way does $\nabla f$ point, and what is its length?
3. In words: divergence measures what? Curl measures what? Which one is a scalar?
4. Can a field have zero divergence but non-zero curl? Give an example. The other way round?
5. What does it mean for a function to satisfy $\nabla^2 f = 0$, in terms of its neighbours?
6. The forward difference $(f(x+h) - f(x))/h$ and the central difference $(f(x+h) - f(x-h))/2h$ both approximate $f'(x)$. Which is more accurate, and by what factor does its error shrink when $h$ is halved?
7. A rigid body spins about the $z$ axis at angular velocity $\omega$. What is the curl of its velocity field?


## 2. The ideas

A derivative is a ratio: a small change in the output of a function divided by the small change in input that caused it. For a function of one variable there is one such ratio, the slope of the graph. For a function of position $f(x, y, z)$ there are three inputs, so there are three ratios, one per direction, and they are called partial derivatives. Nothing new is involved in computing them: hold two variables fixed, differentiate with respect to the third, using every rule you already know. On a computer the function is only known at grid points, so each derivative becomes a difference of neighbouring values divided by the spacing. Which neighbours you use decides the accuracy, and the Taylor expansion tells you exactly how accurate: a one-sided difference has an error proportional to the spacing $h$, a symmetric one has an error proportional to $h^2$. That factor is the single most useful fact about numerical derivatives, because it tells you how fine a grid you need and lets you check that your code is doing what you think.

The four operators of this notebook are particular combinations of partial derivatives. They are not arbitrary. Each one answers a physical question that could be asked with an instrument small enough to sit at a single point, and the formula is what the instrument's reading converges to as it shrinks.

The gradient asks: standing at this point on a scalar field, which way is uphill, and how steep is it? Its answer is a vector, $\nabla f$, pointing in the direction of fastest increase with a length equal to that rate. The gradient is perpendicular to the contour lines, because walking along a contour changes nothing. In electrostatics the potential $V$ is a scalar field and the electric field is $\mathbf{E} = -\nabla V$: the field points downhill in potential, and its strength is the slope. That one relation is why the whole of chapter 3 can work with a scalar instead of a vector.

The divergence asks: if this vector field were the velocity of a fluid, would a tiny box drawn around this point gain fluid or lose it? Count the flow out through each face, add the six faces, divide by the volume of the box, and shrink the box. The answer is a number, $\nabla\cdot\mathbf{F}$, positive at a source and negative at a sink. Gauss' law in chapter 4, $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$, says the sources of the electric field are exactly the charges.

The curl asks: would a tiny paddlewheel placed at this point spin? Add up the field along a tiny closed loop, divide by the loop's area, and shrink the loop. There are three independent planes in which to hold the loop, so the answer has three components and is a vector, $\nabla\times\mathbf{F}$, whose direction is the axis of spin. A field can have curl even where all its arrows are parallel, if the arrows on one side of the wheel are longer than on the other. Ampere's law and Faraday's law in chapters 12 and 14 are statements about the curl of $\mathbf{B}$ and $\mathbf{E}$.

The Laplacian asks: is the value at this point above or below the average of its neighbours? It is the divergence of the gradient, $\nabla^2 f = \nabla\cdot(\nabla f)$, and on a grid it is literally the neighbour average minus the centre value, times a constant. A function whose Laplacian is zero equals the average of its surroundings everywhere, so it has no bumps and no hollows. Poisson's equation in chapter 6, $\nabla^2 V = -\rho/\epsilon_0$, is the gradient and divergence relations above combined into one equation for $V$, and solving it on a grid by repeated neighbour averaging is the whole of module 2's relaxation method.

Two identities tie the operators together and are true for every smooth field: the curl of a gradient is zero, and the divergence of a curl is zero. Both come from the fact that mixed second derivatives do not care about order. They matter because they run in reverse: a field with no curl can always be written as a gradient (that is why an electric potential exists), and a field with no divergence can always be written as a curl (that is why a magnetic vector potential exists). Together with the fact that a field vanishing at infinity is fixed by its divergence and its curl, they explain why Maxwell's equations have the shape they do: two of them give the divergences of $\mathbf{E}$ and $\mathbf{B}$, the other two give the curls, and that is enough to determine both fields.

The main notebooks of module 1 use all of this as finished tools: `emlib.fields.gradient`, `divergence` and `curl` are central differences on `"ij"` grids, and notebook 01 checks them against SymPy. What this primer adds is the reason each formula looks the way it does, and the habit of checking a numerical derivative against its known error rate.


## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=5, suppress=True)

### 3.1 Partial derivatives and finite differences

For $f(x, y, z)$, the partial derivative $\partial f/\partial x$ is the ordinary derivative with respect to $x$ while $y$ and $z$ are held fixed. Rules carry over unchanged: differentiate term by term, treat the other variables as constants. The notation $\partial$ instead of $d$ is only a reminder that other variables exist and are being held still. Shorthands you will meet: $\partial_x f$, $f_x$, and in this repository's code `np.gradient(f, h, axis=0)`.

**Worked example 1.** $f = x^2y + \sin x - y^3$. $\partial f/\partial x = 2xy + \cos x$ (the $y^3$ term has no $x$ and drops out). $\partial f/\partial y = x^2 - 3y^2$ (the $\sin x$ drops out). At $(1, 2)$: $\partial f/\partial x = 4 + 0.5403 = 4.5403$ and $\partial f/\partial y = 1 - 12 = -11$.

Second derivatives are derivatives of derivatives, and there are now mixed ones. The order of a mixed derivative does not matter for any smooth function: $\partial_x\partial_y f = \partial_y\partial_x f$. This small fact is the reason behind both identities in section 3.6.

**Worked example 2 (mixed partials).** From example 1, $\partial_y(\partial_x f) = \partial_y(2xy + \cos x) = 2x$ and $\partial_x(\partial_y f) = \partial_x(x^2 - 3y^2) = 2x$. Equal. A less obvious case: $f = xe^{xy}$. $\partial_x f = e^{xy} + xye^{xy} = e^{xy}(1 + xy)$, then $\partial_y$ of that is $xe^{xy}(1 + xy) + e^{xy}\cdot x = xe^{xy}(2 + xy)$. The other order: $\partial_y f = x^2e^{xy}$, then $\partial_x$ gives $2xe^{xy} + x^2ye^{xy} = xe^{xy}(2 + xy)$. Equal again.

**Worked example 3 (chain rule).** Many fields depend on position only through the distance $r = \sqrt{x^2 + y^2 + z^2}$. Then $\partial r/\partial x = \frac12(x^2 + y^2 + z^2)^{-1/2}\cdot 2x = x/r$, and for any $f = g(r)$, $\partial f/\partial x = g'(r)\,x/r$. For $g = r^3$: $\partial(r^3)/\partial x = 3r^2\cdot x/r = 3rx$. At $(1, 2, 2)$, $r = 3$ and $\partial(r^3)/\partial x = 9$. This chain rule is used every time the Coulomb potential $1/r$ is differentiated.

On a computer we only have $f$ at grid points, so a derivative becomes a difference quotient. There are two obvious choices, and the Taylor expansion says how good each one is. Expand $f$ about $x$:
$$f(x + h) = f + hf' + \tfrac12h^2f'' + \tfrac16h^3f''' + \tfrac1{24}h^4f'''' + \dots, \qquad f(x - h) = f - hf' + \tfrac12h^2f'' - \tfrac16h^3f''' + \tfrac1{24}h^4f'''' - \dots$$
The forward difference rearranges the first line: $\dfrac{f(x+h) - f(x)}{h} = f' + \tfrac12hf'' + \dots$, so its error is about $\tfrac12h|f''|$, proportional to $h$. Subtracting the two lines kills every even term, so the central difference $\dfrac{f(x+h) - f(x-h)}{2h} = f' + \tfrac16h^2f''' + \dots$ has an error about $\tfrac16h^2|f'''|$, proportional to $h^2$. Halve $h$ and the forward error halves while the central error drops by four. Adding the two lines kills the odd terms and gives the second derivative:
$$\frac{f(x+h) - 2f(x) + f(x-h)}{h^2} = f'' + \tfrac1{12}h^2f'''' + \dots,$$
also second order. This three-point formula is the Laplacian's building block in section 3.5 and the stencil of every relaxation solver in module 2. `np.gradient` uses central differences inside an array and one-sided ones at the two edges, which is why the edge rows of any numerical derivative in these notebooks are less accurate than the interior.

**Worked example 4 (predicting the error).** For example 1 at $(1, 2)$, differentiating in $x$: $f_{xx} = 2y - \sin x = 3.1585$, $f_{xxx} = -\cos x = -0.5403$, $f_{xxxx} = \sin x = 0.8415$. With $h = 0.1$ the predictions are: forward error $\tfrac12\cdot0.1\cdot3.1585 = 0.158$, central error $\tfrac16\cdot0.01\cdot0.5403 = 0.00090$, second-derivative error $\tfrac1{12}\cdot0.01\cdot0.8415 = 0.00070$. The table below shows the forward and central errors; the next cell compares all three with these predictions.


In [ ]:
f = lambda x, y: x**2 * y + np.sin(x) - y**3
fx_exact = lambda x, y: 2 * x * y + np.cos(x)
x0, y0 = 1.0, 2.0
print(f"exact df/dx at (1, 2) = {fx_exact(x0, y0):.6f}      exact df/dy = {x0**2 - 3 * y0**2}")
print("      h      forward error    central error")
for h in (0.1, 0.01, 0.001, 0.0001):
    fwd = (f(x0 + h, y0) - f(x0, y0)) / h
    cen = (f(x0 + h, y0) - f(x0 - h, y0)) / (2 * h)
    print(f"{h:8.4f}   {abs(fwd - fx_exact(x0, y0)):12.2e}   {abs(cen - fx_exact(x0, y0)):12.2e}")
print("forward: error falls 10x per 10x smaller h (first order). central: 100x (second order).")

In [ ]:
# Worked example 4: the Taylor-series error predictions against the measured errors.
fxx = 2 * y0 - np.sin(x0); fxxx = -np.cos(x0); fxxxx = np.sin(x0)
print("      h     forward: measured  predicted      central: measured  predicted      2nd deriv: measured  predicted")
for h in (0.1, 0.01):
    fwd = (f(x0 + h, y0) - f(x0, y0)) / h
    cen = (f(x0 + h, y0) - f(x0 - h, y0)) / (2 * h)
    sec = (f(x0 + h, y0) - 2 * f(x0, y0) + f(x0 - h, y0)) / h**2
    print(f"{h:8.3f}     {abs(fwd - fx_exact(x0, y0)):10.5f}  {h / 2 * abs(fxx):10.5f}      {abs(cen - fx_exact(x0, y0)):12.2e}  {h**2 / 6 * abs(fxxx):10.2e}      {abs(sec - fxx):12.2e}  {h**2 / 12 * abs(fxxxx):10.2e}")

# Worked examples 2 and 3 with SymPy, which does the algebra exactly.
import sympy as sp
xs, ys, zs = sp.symbols("x y z")
g = xs * sp.exp(xs * ys)
print("mixed partials of x e^(xy):", sp.simplify(sp.diff(g, xs, ys) - sp.diff(g, ys, xs)) == 0, "(difference simplifies to zero)")
r_sym = sp.sqrt(xs**2 + ys**2 + zs**2)
print("d(r^3)/dx at (1, 2, 2) =", sp.diff(r_sym**3, xs).subs({xs: 1, ys: 2, zs: 2}), "   chain rule 3 r x = 9")


### 3.2 The gradient

Put the partial derivatives of a scalar field together as a vector:
$$\nabla f = \frac{\partial f}{\partial x}\hat{\mathbf{x}} + \frac{\partial f}{\partial y}\hat{\mathbf{y}} + \frac{\partial f}{\partial z}\hat{\mathbf{z}}.$$

Why this vector? Move from $\mathbf{r}$ by a small step $d\mathbf{l} = (dx, dy, dz)$. The change in $f$ is, to first order,
$$df = \frac{\partial f}{\partial x}dx + \frac{\partial f}{\partial y}dy + \frac{\partial f}{\partial z}dz = \nabla f\cdot d\mathbf{l}.$$
That dot product tells the whole story. For a step of fixed length, $df$ is largest when $d\mathbf{l}$ is parallel to $\nabla f$: the gradient points in the direction of steepest increase. Its length is the slope in that direction. Stepping perpendicular to $\nabla f$ gives $df = 0$, which means you are walking along a contour: the gradient is perpendicular to the contour lines. And the slope in an arbitrary unit direction $\hat{\mathbf{u}}$, the directional derivative, is $\nabla f\cdot\hat{\mathbf{u}}$, which is never larger than $|\nabla f|$ and reaches it only along the gradient itself.

Two pitfalls. First, the gradient of a scalar field is a vector field: one arrow at every point, not one arrow for the whole field. Second, $\nabla f$ points uphill, so a particle that rolls downhill moves along $-\nabla f$. The minus sign in $\mathbf{E} = -\nabla V$ is that: positive charges are pushed from high potential to low. In chapter 3, $\mathbf{E} = -\nabla V$ means the electric field points downhill in potential, perpendicular to the equipotential surfaces, with a strength equal to the slope. Closely spaced equipotentials mean a strong field.

**Worked example 5.** Bowl $f = \tfrac12x^2 + y^2 - \tfrac12xy$: $\nabla f = (x - \tfrac12y,\; 2y - \tfrac12x)$. At $(1, 0.5)$: $(0.75, 0.5)$, length $0.901$. Slope towards $(1, 1)/\sqrt2$: $(0.75 + 0.5)/\sqrt2 = 0.884$. Slope towards $(1, 0)$: $0.75$. Slope along the contour direction, perpendicular to the gradient, $(-0.5, 0.75)/0.901$: $(-0.375 + 0.375)/0.901 = 0$.

**Worked example 6 (directional derivative).** $f = x^2y$ at $(2, 1)$, moving in the direction $(3, 4)$. First make the direction a unit vector: $\hat{\mathbf{u}} = (0.6, 0.8)$. $\nabla f = (2xy, x^2) = (4, 4)$. Slope $= 4\cdot0.6 + 4\cdot0.8 = 5.6$. The steepest slope at this point is $|\nabla f| = 4\sqrt2 = 5.657$, in the direction $(1, 1)/\sqrt2$, so $(3, 4)$ is close to the steepest direction but not quite on it. Forgetting to normalise the direction is the usual mistake: with $(3, 4)$ itself the dot product gives $28$, which is a rate per five units of distance, not per one.

**Worked example 7 (gradients of $r$ and of products).** By the chain rule of example 3, $\nabla r = (x, y, z)/r = \hat{\mathbf{r}}$: the distance from the origin increases fastest straight outward, at a rate of one. Then $\nabla(r^n) = nr^{n-1}\hat{\mathbf{r}}$ for any $n$, so $\nabla(r^2) = 2\mathbf{r}$ and $\nabla(1/r) = -\hat{\mathbf{r}}/r^2$. The product rule holds as for ordinary derivatives: $\nabla(fg) = f\nabla g + g\nabla f$. Check on $xy^2$: directly, $\nabla(xy^2) = (y^2, 2xy)$; by the rule with $f = x$, $g = y^2$: $x(0, 2y) + y^2(1, 0) = (y^2, 2xy)$.

**Worked example 8 (the Coulomb field from a potential).** $f = 1/r$ with $r = \sqrt{x^2 + y^2 + z^2}$. From example 7, $\nabla(1/r) = -\hat{\mathbf{r}}/r^2$. The potential of a point charge is $V = \dfrac{q}{4\pi\epsilon_0 r}$ (chapter 3), so
$$\mathbf{E} = -\nabla V = \frac{q}{4\pi\epsilon_0}\frac{\hat{\mathbf{r}}}{r^2},$$
Coulomb's law falls out of the gradient. At $(1, 2, 2)$, $r = 3$ and $-\nabla(1/r) = (1, 2, 2)/27 = (0.037, 0.074, 0.074)$, with magnitude $1/9$. Notebook 03 of this module computes $V$ on a grid for many charges and recovers $\mathbf{E}$ in exactly this way.


In [ ]:
x = np.linspace(-2, 2, 81); y = np.linspace(-2, 2, 81)
X, Y = np.meshgrid(x, y, indexing="ij"); h = x[1] - x[0]
bowl = X**2 / 2 + Y**2 - X * Y / 2
gx, gy = np.gradient(bowl, h, h)

fig, ax = plt.subplots(figsize=(6.5, 6))
cs = ax.contour(X, Y, bowl, levels=15, cmap="viridis"); ax.clabel(cs, fontsize=7)
st = 6
ax.quiver(X[::st, ::st], Y[::st, ::st], gx[::st, ::st], gy[::st, ::st], color="k")
ax.set_aspect("equal"); ax.set_title("gradient arrows: perpendicular to contours, pointing uphill, longer where steeper"); plt.show()

p = np.array([1.0, 0.5]); grad_p = np.array([p[0] - p[1] / 2, 2 * p[1] - p[0] / 2])
bowl_f = lambda q: q[0]**2 / 2 + q[1]**2 - q[0] * q[1] / 2
for name, u in (("(1,1)/sqrt2", np.array([1, 1]) / np.sqrt(2)), ("(1,0)", np.array([1.0, 0])), ("along contour", np.array([-0.5, 0.75]) / 0.901)):
    e = 1e-5
    print(f"slope in direction {name:14s}: grad . u = {grad_p @ u:+.4f}   direct finite difference = {(bowl_f(p + e * u) - bowl_f(p - e * u)) / (2 * e):+.4f}")

In [ ]:
# Worked examples 6, 7 and 8.
grad_f6 = np.array([2 * 2 * 1, 2**2])                       # grad(x^2 y) at (2, 1)
u = np.array([3.0, 4.0]) / 5
print(f"example 6: grad = {grad_f6}, slope along (3,4)/5 = {grad_f6 @ u:.3f}, steepest slope |grad| = {np.linalg.norm(grad_f6):.3f}, un-normalised dot product = {grad_f6 @ np.array([3.0, 4.0]):.0f}")

grad_sym = lambda expr: [sp.diff(expr, v) for v in (xs, ys, zs)]
print("example 7: grad(1/r) =", [sp.simplify(c) for c in grad_sym(1 / r_sym)], "  = -(x, y, z)/r^3")
print("           grad(x y^2) =", grad_sym(xs * ys**2)[:2], "  product rule x grad(y^2) + y^2 grad(x) =", [sp.expand(xs * sp.diff(ys**2, v) + ys**2 * sp.diff(xs, v)) for v in (xs, ys)])

p = np.array([1.0, 2.0, 2.0]); r = np.linalg.norm(p)
E_from_V = p / r**3                                           # -grad(1/r), in units of q/(4 pi eps0)
hh = 1e-5
E_fd = np.array([-( 1 / np.linalg.norm(p + hh * e) - 1 / np.linalg.norm(p - hh * e)) / (2 * hh) for e in np.eye(3)])
print(f"example 8: -grad(1/r) at (1, 2, 2) by hand = {E_from_V}, by finite differences = {E_fd}, magnitude 1/r^2 = {1 / r**2:.4f}")


A gradient has a use beyond physics that makes its meaning stick: walking downhill. Starting anywhere, take repeated steps of $-\alpha\nabla f$ and you descend towards a minimum. The path crosses every contour at a right angle. (This is gradient descent, the workhorse of optimisation.)

In [ ]:
grad_bowl = lambda q: np.array([q[0] - q[1] / 2, 2 * q[1] - q[0] / 2])
path = [np.array([1.8, 1.5])]
for _ in range(25):
    path.append(path[-1] - 0.25 * grad_bowl(path[-1]))
path = np.array(path)
fig, ax = plt.subplots(figsize=(6, 6))
ax.contour(X, Y, bowl, levels=20, cmap="viridis")
ax.plot(path[:, 0], path[:, 1], "o-", color="C3", ms=4, label="steps of -0.25 grad f")
ax.set_aspect("equal"); ax.legend(); ax.set_title("walking downhill with the gradient"); plt.show()
print("final point", path[-1], "  minimum is at (0, 0)")

### 3.3 The divergence

Imagine a vector field as the velocity of a fluid. Draw a tiny box of sides $\Delta x, \Delta y, \Delta z$ around a point. Fluid crosses each of the six faces, and only the component of velocity normal to a face carries anything through it. Through the right face (normal $+\hat{\mathbf{x}}$, area $\Delta y\Delta z$) the outward flow is $F_x(x + \Delta x/2)\,\Delta y\Delta z$; through the left face (normal $-\hat{\mathbf{x}}$) it is $-F_x(x - \Delta x/2)\,\Delta y\Delta z$. Their sum is
$$\big[F_x(x + \tfrac12\Delta x) - F_x(x - \tfrac12\Delta x)\big]\Delta y\Delta z \approx \frac{\partial F_x}{\partial x}\Delta x\Delta y\Delta z.$$
The top and bottom faces (normals $\pm\hat{\mathbf{y}}$, area $\Delta x\Delta z$) give $\dfrac{\partial F_y}{\partial y}\Delta x\Delta y\Delta z$ by the same argument, and the front and back faces give $\dfrac{\partial F_z}{\partial z}\Delta x\Delta y\Delta z$. Each pair contributes a derivative of its own component along its own direction, and nothing else: $F_y$ never crosses an $x$ face. Divide the total outward flux by the box volume $\Delta x\Delta y\Delta z$ and let the box shrink:
$$\nabla\cdot\mathbf{F} = \lim_{\Delta v\to0}\frac{\text{flux out of the box}}{\Delta v} = \frac{\partial F_x}{\partial x} + \frac{\partial F_y}{\partial y} + \frac{\partial F_z}{\partial z}.$$

So divergence is outward flux per unit volume: positive where the field is being created (a source, fluid appearing), negative where it is being destroyed (a sink), zero where whatever enters also leaves. For a real fluid, the divergence of the velocity is the fractional rate at which a small blob of fluid expands: $\nabla\cdot\mathbf{v} = 0$ is the statement that water is incompressible. In chapter 4, $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ says that electric charge is exactly where electric field lines start and end, and $\nabla\cdot\mathbf{B} = 0$ in chapter 12 says magnetic field lines never start or end anywhere.

A pitfall: divergence is not "the field spreads out in the picture". The Coulomb field spreads out visibly, yet its divergence is zero everywhere except at the charge (example 12). What matters is the balance of flux through a tiny box, and for a $1/r^2$ field the growth of the box's outer face exactly compensates the fall of the field.

**Worked example 9.** The six reference fields from notebook 00a, in 2D ($\nabla\cdot\mathbf{F} = \partial F_x/\partial x + \partial F_y/\partial y$):

| field | $F_x$ | $F_y$ | $\partial F_x/\partial x$ | $\partial F_y/\partial y$ | div |
|---|---|---|---|---|---|
| uniform | 1 | 0 | 0 | 0 | 0 |
| radial | $x$ | $y$ | 1 | 1 | 2 |
| rotation | $-y$ | $x$ | 0 | 0 | 0 |
| saddle | $x$ | $-y$ | 1 | $-1$ | 0 |
| shear | $y$ | 0 | 0 | 0 | 0 |
| sink+swirl | $-x-y$ | $x-y$ | $-1$ | $-1$ | $-2$ |

**Worked example 10.** $\mathbf{F} = (x^2, xy, z)$ in 3D: $\nabla\cdot\mathbf{F} = 2x + x + 1 = 3x + 1$. It depends on position: a source for $x > -1/3$, a sink for $x < -1/3$.

**Worked example 11 (at a point).** $\mathbf{F} = (xyz,\ y^2,\ \sin(xz))$ at $(1, 2, \pi)$. $\partial F_x/\partial x = yz = 2\pi = 6.283$. $\partial F_y/\partial y = 2y = 4$. $\partial F_z/\partial z = x\cos(xz) = \cos\pi = -1$. Sum: $9.283$. Only three of the nine possible partial derivatives of a 3D vector field enter the divergence; the other six belong to the curl.

**Worked example 12 (the Coulomb field).** $\mathbf{F} = \mathbf{r}/r^3 = (x, y, z)/r^3$. With the chain rule, $\dfrac{\partial}{\partial x}\dfrac{x}{r^3} = \dfrac{1}{r^3} - \dfrac{3x\cdot x}{r^5}$. Add the three such terms: $\dfrac{3}{r^3} - \dfrac{3(x^2 + y^2 + z^2)}{r^5} = \dfrac{3}{r^3} - \dfrac{3}{r^3} = 0$. The field of a point charge has no sources anywhere except at the charge itself, where $r = 0$ and the calculation breaks down. Notebook 00d returns to what happens there.

The code measures the divergence the way it was defined, as flux out of a shrinking square divided by its area, and compares with the table; then it checks examples 11 and 12.


**Animation.** The two instruments of sections 3.3 and 3.4, shrunk onto a point. Left: the flux out of a square around $(0.5, 0.4)$ divided by its area, for $\mathbf{F} = (x^3, xy^2)$, whose divergence there is $3x^2 + 2xy = 1.15$. Right: the circulation around a square loop at $(0.3, -0.4)$ divided by its area, for $\mathbf{F} = (-y^3, x^3)$, whose curl there is $3x^2 + 3y^2 = 0.75$. Red arrows mark field leaving the box or running with the loop, blue the opposite. As the half-width $h$ falls from 1 to 0.01 both ratios close in on the exact values, with an error proportional to $h^2$. The fields are cubic on purpose: for a field built from quadratics, like $(x^2, xy)$ in the cell below, the symmetric square cancels the error terms and the ratio is exact at every size, so there would be nothing to watch.

In [ ]:
from emlib import show
show.animation("m1_fields", "ShrinkingInstruments")

In [ ]:
def flux_out_of_square(Fx, Fy, cx, cy, half, n=200):
    """Outward flux of (Fx, Fy) through the square of half-width half centred at (cx, cy)."""
    s = cx + np.linspace(-half, half, n); t = cy + np.linspace(-half, half, n)
    ds = s[1] - s[0]
    right  = np.trapezoid(Fx(np.full_like(t, cx + half), t), dx=ds)
    left   = np.trapezoid(-Fx(np.full_like(t, cx - half), t), dx=ds)
    top    = np.trapezoid(Fy(s, np.full_like(s, cy + half)), dx=ds)
    bottom = np.trapezoid(-Fy(s, np.full_like(s, cy - half)), dx=ds)
    return right + left + top + bottom

fields_2d = {
    "uniform":    (lambda x, y: np.ones_like(x), lambda x, y: np.zeros_like(x)),
    "radial":     (lambda x, y: x, lambda x, y: y),
    "rotation":   (lambda x, y: -y, lambda x, y: x),
    "saddle":     (lambda x, y: x, lambda x, y: -y),
    "shear":      (lambda x, y: y, lambda x, y: np.zeros_like(x)),
    "sink+swirl": (lambda x, y: -x - y, lambda x, y: x - y),
}
print("flux / area of shrinking squares centred at (0.7, -0.4):")
for name, (Fx, Fy) in fields_2d.items():
    row = [flux_out_of_square(Fx, Fy, 0.7, -0.4, half) / (2 * half) ** 2 for half in (1.0, 0.1, 0.01)]
    print(f"  {name:11s} {np.round(row, 4)}")

Fx5, Fy5 = (lambda x, y: x**2), (lambda x, y: x * y)      # the 2D part of worked example 5: div = 2x + x = 3x
for cx in (-1.0, 0.0, 0.5):
    print(f"  (x^2, xy) at x = {cx:+.1f}: measured {flux_out_of_square(Fx5, Fy5, cx, 0.3, 0.001) / 0.002**2:.4f}   formula 3x = {3 * cx:+.1f}")

In [ ]:
# Worked examples 11 and 12 with SymPy.
div_sym = lambda F: sum(sp.diff(F[k], v) for k, v in enumerate((xs, ys, zs)))
F11 = [xs * ys * zs, ys**2, sp.sin(xs * zs)]
print("example 11: div F =", div_sym(F11), "   at (1, 2, pi):", sp.N(div_sym(F11).subs({xs: 1, ys: 2, zs: sp.pi}), 5))
F12 = [xs / r_sym**3, ys / r_sym**3, zs / r_sym**3]
print("example 12: div(r / r^3) =", sp.simplify(div_sym(F12)), "   (zero for r > 0)")

# Divergence of velocity as the expansion rate of a blob: advect a small circle of fluid markers in the radial field (div = 2)
# for a short time dt and compare the area ratio with 1 + 2 dt.
t = np.linspace(0, 2 * np.pi, 200, endpoint=False)
blob = np.stack([0.7 + 0.05 * np.cos(t), -0.4 + 0.05 * np.sin(t)], axis=-1)
dt = 0.01
moved = blob + dt * blob                                              # velocity (x, y) at each marker
area = lambda poly: 0.5 * abs(np.sum(poly[:, 0] * np.roll(poly[:, 1], -1) - np.roll(poly[:, 0], -1) * poly[:, 1]))
print(f"radial field: blob area grows by a factor {area(moved) / area(blob):.4f} in dt = {dt};  1 + div * dt = {1 + 2 * dt:.4f}")


### 3.4 The curl

Now the other instrument: a tiny paddlewheel, or equivalently a tiny closed loop. Take a square loop of sides $\Delta x, \Delta y$ in the $xy$ plane, traversed counter-clockwise, and add up $\mathbf{F}\cdot d\mathbf{l}$ around it. Along the bottom edge (moving in $+x$, at height $y - \Delta y/2$) the contribution is $F_x(y - \Delta y/2)\Delta x$; along the top edge (moving in $-x$) it is $-F_x(y + \Delta y/2)\Delta x$. Together: $-\frac{\partial F_x}{\partial y}\Delta x\Delta y$. The right and left edges give $+\frac{\partial F_y}{\partial x}\Delta x\Delta y$. Divide by the area and shrink:
$$(\nabla\times\mathbf{F})_z = \lim_{\Delta S\to0}\frac{\text{circulation around the loop}}{\Delta S} = \frac{\partial F_y}{\partial x} - \frac{\partial F_x}{\partial y}.$$

Loops in the other two planes give the $x$ and $y$ components, with the indices cycling ($x \to yz$, $y \to zx$):
$$\nabla\times\mathbf{F} = \left(\frac{\partial F_z}{\partial y} - \frac{\partial F_y}{\partial z}\right)\hat{\mathbf{x}} + \left(\frac{\partial F_x}{\partial z} - \frac{\partial F_z}{\partial x}\right)\hat{\mathbf{y}} + \left(\frac{\partial F_y}{\partial x} - \frac{\partial F_x}{\partial y}\right)\hat{\mathbf{z}}.$$
Same pattern as the cross product in 00a with $\nabla$ as the first vector, which is why it is written $\nabla\times\mathbf{F}$. Each component uses the two derivatives that the divergence left out in its plane: the divergence takes $\partial F_x/\partial x$, the $z$ curl takes $\partial F_x/\partial y$ and $\partial F_y/\partial x$.

So curl is circulation per unit area, a vector whose direction is the axis a paddlewheel would spin about (right-hand rule: fingers along the circulation, thumb along the curl) and whose length is twice the spin rate. In chapter 14, $\nabla\times\mathbf{E} = -\partial\mathbf{B}/\partial t$ says a changing magnetic field makes the electric field circulate. In chapter 12, $\nabla\times\mathbf{B} = \mu_0\mathbf{J}$ says current makes the magnetic field circulate around it.

Curl is a local property, and a picture of arrows can hide it. The shear field below has every arrow parallel and still has curl, because the paddlewheel feels the difference in speed across its diameter, not the direction of the flow. Conversely the wire field $\hat{\boldsymbol\phi}/r$ in the Explore section visibly circulates around the origin and has zero curl at every point away from it. Trust the loop, not the picture.

**Worked example 13.** The reference fields, $z$ component $\partial F_y/\partial x - \partial F_x/\partial y$:

| field | $F_x$ | $F_y$ | $\partial F_y/\partial x$ | $\partial F_x/\partial y$ | curl$_z$ |
|---|---|---|---|---|---|
| uniform | 1 | 0 | 0 | 0 | 0 |
| radial | $x$ | $y$ | 0 | 0 | 0 |
| rotation | $-y$ | $x$ | 1 | $-1$ | 2 |
| saddle | $x$ | $-y$ | 0 | 0 | 0 |
| shear | $y$ | 0 | 0 | 1 | $-1$ |
| sink+swirl | $-x-y$ | $x-y$ | 1 | $-1$ | 2 |

The shear field, all arrows parallel, has curl $-1$: a paddlewheel in it spins clockwise because the flow is faster on its upper side. The sink-with-swirl field has both divergence $-2$ and curl $2$. Divergence and curl are independent: any combination is possible.

**Worked example 14 (rigid rotation).** A body spinning about the $z$ axis at angular velocity $\omega$ has velocity $\mathbf{v} = \boldsymbol{\omega}\times\mathbf{r} = (-\omega y, \omega x, 0)$, the rotation field scaled by $\omega$. Its $z$ curl is $\partial_x(\omega x) - \partial_y(-\omega y) = \omega + \omega = 2\omega$, and the other components vanish. In general $\nabla\times(\boldsymbol\omega\times\mathbf{r}) = 2\boldsymbol\omega$, which is where "curl is twice the spin rate" comes from: a paddlewheel carried along by a rigidly rotating fluid turns once per revolution of the fluid, and the curl reports $2\omega$ for that.

**Worked example 15.** $\mathbf{F} = (yz, zx, xy)$ in 3D. $x$: $\partial F_z/\partial y - \partial F_y/\partial z = x - x = 0$. $y$: $\partial F_x/\partial z - \partial F_z/\partial x = y - y = 0$. $z$: $\partial F_y/\partial x - \partial F_x/\partial y = z - z = 0$. Curl zero everywhere. Indeed $\mathbf{F} = \nabla(xyz)$, and the curl of a gradient is always zero (section 3.6).

**Worked example 16 (all three components).** $\mathbf{F} = (y^2z,\ xz^2,\ xy)$ at $(1, 2, 3)$. $x$: $\partial_y(xy) - \partial_z(xz^2) = x - 2xz = 1 - 6 = -5$. $y$: $\partial_z(y^2z) - \partial_x(xy) = y^2 - y = 4 - 2 = 2$. $z$: $\partial_x(xz^2) - \partial_y(y^2z) = z^2 - 2yz = 9 - 12 = -3$. So $\nabla\times\mathbf{F} = (-5, 2, -3)$ there: a paddlewheel at that point spins about an axis pointing mostly along $-x$.


**Animation.** A field morphs from the pure rotation $(-y, x)$ to the pure expansion $(x, y)$, and the readout shows $\nabla\cdot\mathbf{F} = 2s$ and $(\nabla\times\mathbf{F})_z = 2(1 - s)$ as the blend parameter $s$ goes from 0 to 1. The paddlewheel spins at half the curl, so it turns steadily at the start and stops once the curl is gone. Near the centre the arrows barely seem to change, which is the warning of this section: curl and divergence are local properties that a quick look at the arrows can miss. The right half of the ShrinkingInstruments animation in section 3.3 measures a curl with a shrinking loop, which is exactly the definition used in this section.

In [ ]:
from emlib import show
show.animation("m1_fields", "DivergenceVersusCurl")

In [ ]:
def circulation_around_square(Fx, Fy, cx, cy, half, n=200):
    """Counter-clockwise line integral of (Fx, Fy) around the square of half-width half centred at (cx, cy)."""
    s = np.linspace(-half, half, n); ds = s[1] - s[0]
    bottom = np.trapezoid(Fx(cx + s, np.full_like(s, cy - half)), dx=ds)      # moving +x
    right  = np.trapezoid(Fy(np.full_like(s, cx + half), cy + s), dx=ds)      # moving +y
    top    = np.trapezoid(-Fx(cx + s, np.full_like(s, cy + half)), dx=ds)     # moving -x
    left   = np.trapezoid(-Fy(np.full_like(s, cx - half), cy + s), dx=ds)     # moving -y
    return bottom + right + top + left

print("circulation / area of shrinking loops centred at (0.7, -0.4):")
for name, (Fx, Fy) in fields_2d.items():
    row = [circulation_around_square(Fx, Fy, 0.7, -0.4, half) / (2 * half) ** 2 for half in (1.0, 0.1, 0.01)]
    print(f"  {name:11s} {np.round(row, 4)}")

# The paddlewheel picture for the shear field: the flow is faster above the wheel than below it.
fig, ax = plt.subplots(figsize=(6, 4))
yy = np.linspace(-1, 1, 9)
ax.quiver(np.zeros_like(yy) - 1.5, yy, yy, np.zeros_like(yy), scale=6, color="grey")
ax.add_patch(plt.Circle((0, 0.4), 0.3, fill=False, lw=2, color="C3"))
ax.annotate("", xy=(0.3, 0.4), xytext=(0.28, 0.55), arrowprops=dict(arrowstyle="->", color="C3", lw=2))
ax.text(0.4, 0.7, "faster flow on top pushes\nthe wheel clockwise: curl_z = -1", color="C3")
ax.set_xlim(-2, 2.5); ax.set_ylim(-1.2, 1.2); ax.set_aspect("equal"); ax.set_title("shear field F = (y, 0)"); ax.axis("off"); plt.show()

In [ ]:
# Worked examples 14 and 16 with SymPy.
curl_sym = lambda F: [sp.diff(F[2], ys) - sp.diff(F[1], zs), sp.diff(F[0], zs) - sp.diff(F[2], xs), sp.diff(F[1], xs) - sp.diff(F[0], ys)]
w = sp.symbols("omega")
print("example 14: curl of (-w y, w x, 0) =", curl_sym([-w * ys, w * xs, 0]), "  = 2 omega z-hat")
F16 = [ys**2 * zs, xs * zs**2, xs * ys]
print("example 16: curl F =", curl_sym(F16), "   at (1, 2, 3):", [c.subs({xs: 1, ys: 2, zs: 3}) for c in curl_sym(F16)])


### 3.5 The Laplacian

The Laplacian is the divergence of the gradient, $\nabla^2 f = \nabla\cdot(\nabla f) = \partial^2 f/\partial x^2 + \partial^2 f/\partial y^2 + \partial^2 f/\partial z^2$. It appears when the two first-order operators are combined, which happens in chapter 6: $\mathbf{E} = -\nabla V$ and $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ give $\nabla^2 V = -\rho/\epsilon_0$, Poisson's equation.

Start in one dimension, where $\nabla^2 f$ is just $f''$, the curvature of the graph. Where $f'' > 0$ the graph curves upward and every point sits below the chord joining its neighbours; where $f'' < 0$ it curves downward and sits above; where $f'' = 0$ the graph is a straight line and each point is exactly the average of its two neighbours. That last reading is the one to keep, because it survives into higher dimensions and onto a grid. The second difference $(f_{i+1} - 2f_i + f_{i-1})/h^2$ from section 3.1 approximates $\partial^2 f/\partial x^2$; it can be rewritten as $\frac{2}{h^2}\big(\tfrac12(f_{i+1} + f_{i-1}) - f_i\big)$, the neighbour average minus the centre. Adding the same for $y$ gives
$$\nabla^2 f \approx \frac{f_{i+1,j} + f_{i-1,j} + f_{i,j+1} + f_{i,j-1} - 4f_{i,j}}{h^2} = \frac{4}{h^2}\big(\text{average of the four neighbours} - f_{i,j}\big).$$
So the Laplacian measures how far a point sits below the average of its surroundings. A function with $\nabla^2 f = 0$ (harmonic) equals the average of its neighbours everywhere; it has no bumps or hollows, only saddles. Its value at the centre of any circle or sphere equals its average over that circle or sphere (the mean-value property, checked in the code). That is why the electrostatic potential in empty space has no local maxima or minima, and it is why relaxation methods in module 2 work: keep replacing each value by its neighbour average until nothing changes.

The same equation turns up wherever something flows down a gradient and is conserved. Heat flows along $-\nabla T$ and, in a steady state with no heat sources, $\nabla^2 T = 0$. A dissolved substance diffuses along $-\nabla c$ and, in a steady state, $\nabla^2 c = 0$. The electrostatic potential obeys Laplace's equation in charge-free space for the same structural reason: $\mathbf{E}$ is a gradient and its divergence is fixed by the sources. Anything you learn about solving one of these applies to all of them.

**Worked example 17.** Bowl $f = \tfrac12x^2 + y^2 - \tfrac12xy$: $f_{xx} = 1$, $f_{yy} = 2$, $\nabla^2 f = 3 > 0$. The bowl sits below its neighbours everywhere (it curves up). Saddle $f = x^2 - y^2$: $2 - 2 = 0$, harmonic. Hill $f = -x^2 - y^2$: $-4$, above its neighbours.

**Worked example 18 (radial functions).** In 3D, $r^2 = x^2 + y^2 + z^2$ has $\partial^2(r^2)/\partial x^2 = 2$, so $\nabla^2(r^2) = 6$; in 2D the same function gives $4$. In 2D, $f = \ln r$: $\partial_x\ln r = x/r^2$, then $\partial_{xx}\ln r = 1/r^2 - 2x^2/r^4$, and summing with the $y$ term gives $2/r^2 - 2(x^2 + y^2)/r^4 = 0$ for $r > 0$. So $\ln r$ is the harmonic radial function of the plane, which is why the potential of a long line of charge is a logarithm (module 2, notebook 03).

**Worked example 19.** $f = 1/r$ in 3D away from the origin. From example 7, $\partial f/\partial x = -x/r^3$. Differentiate again: $\partial^2 f/\partial x^2 = -1/r^3 + 3x^2/r^5$. Sum over $x, y, z$: $-3/r^3 + 3(x^2 + y^2 + z^2)/r^5 = -3/r^3 + 3/r^3 = 0$. The Coulomb potential is harmonic everywhere except at the charge. (In 2D the harmonic radial function is $\ln r$, not $1/r$; the two match the two space dimensions in which field lines can spread.)


In [ ]:
def laplacian_grid(F, h):
    return (F[2:, 1:-1] + F[:-2, 1:-1] + F[1:-1, 2:] + F[1:-1, :-2] - 4 * F[1:-1, 1:-1]) / h**2

for name, F, exact in (("bowl", bowl, 3.0), ("saddle x^2 - y^2", X**2 - Y**2, 0.0), ("hill -x^2 - y^2", -X**2 - Y**2, -4.0)):
    L = laplacian_grid(F, h)
    print(f"{name:18s} Laplacian on the grid: mean {L.mean():+.4f}, spread {L.std():.1e}   exact {exact:+.1f}")

# Mean-value property: average of a harmonic function over a circle equals its centre value.
t = np.linspace(0, 2 * np.pi, 2001)[:-1]
for name, func, centre in (("saddle (harmonic)", lambda x, y: x**2 - y**2, (0.6, 0.3)), ("bowl (not harmonic)", lambda x, y: x**2 / 2 + y**2 - x * y / 2, (0.6, 0.3))):
    cx, cy = centre
    avg = np.mean(func(cx + 0.5 * np.cos(t), cy + 0.5 * np.sin(t)))
    print(f"{name:20s} average on a circle of radius 0.5: {avg:.4f}   value at the centre: {func(cx, cy):.4f}")

# Laplacian of 1/r in 3D, away from the origin.
s = np.linspace(1, 2, 41); hs = s[1] - s[0]
X3, Y3, Z3 = np.meshgrid(s, s, s, indexing="ij")
inv_r = 1 / np.sqrt(X3**2 + Y3**2 + Z3**2)
lap3 = sum(np.gradient(np.gradient(inv_r, hs, axis=a), hs, axis=a) for a in range(3))
print(f"Laplacian of 1/r on a 3D grid away from the origin: max |value| = {np.abs(lap3[2:-2, 2:-2, 2:-2]).max():.1e}  (typical |1/r| there ~ {inv_r.mean():.2f})")

In [ ]:
# Worked example 18: the 1D curvature picture and the radial Laplacians.
xx = np.linspace(-1, 1, 11); hx = xx[1] - xx[0]
for name, func in (("x^2 (curves up)", xx**2), ("-x^2 (curves down)", -xx**2), ("2x + 1 (straight)", 2 * xx + 1)):
    second = (func[2:] - 2 * func[1:-1] + func[:-2]) / hx**2
    avg_minus_centre = 0.5 * (func[2:] + func[:-2]) - func[1:-1]
    print(f"{name:20s}: f'' on the grid = {second.mean():+.3f},  neighbour average - centre = {avg_minus_centre.mean():+.4f} (same sign)")

lap_sym = lambda expr, vars_: sum(sp.diff(expr, v, 2) for v in vars_)
r2d = sp.sqrt(xs**2 + ys**2)
print("Laplacian of r^2 in 3D:", lap_sym(r_sym**2, (xs, ys, zs)), "   in 2D:", lap_sym(r2d**2, (xs, ys)))
print("Laplacian of ln r in 2D:", sp.simplify(lap_sym(sp.log(r2d), (xs, ys))), "   of 1/r in 3D:", sp.simplify(lap_sym(1 / r_sym, (xs, ys, zs))), "   (both zero away from the origin)")


### 3.6 Two identities that are always true

$$\nabla\times(\nabla f) = 0 \qquad\text{and}\qquad \nabla\cdot(\nabla\times\mathbf{F}) = 0.$$

Both follow from the fact that mixed partial derivatives commute (worked example 2). For the first: the $z$ component of $\nabla\times(\nabla f)$ is $\partial_x(\partial_y f) - \partial_y(\partial_x f) = 0$, and likewise for $x$ and $y$. For the second, write out $\nabla\cdot(\nabla\times\mathbf{F})$: every mixed derivative such as $\partial_x\partial_y F_z$ appears twice with opposite signs and cancels.

In words: a gradient field never circulates, and a curl field never has sources. Picture the gradient of a landscape: walking around any closed loop brings you back to the same height, so the net uphill is zero, so the circulation is zero. Picture the curl as the swirl of a fluid: swirl has no beginning and no end, so no flux of it leaves a closed box.

They matter because they run backwards. A field with zero curl is the gradient of some scalar (that is how the electric potential $V$ exists in chapter 3: $\nabla\times\mathbf{E} = 0$ in electrostatics, so $\mathbf{E} = -\nabla V$). A field with zero divergence is the curl of some vector (that is how the magnetic vector potential $\mathbf{A}$ exists in chapter 11: $\nabla\cdot\mathbf{B} = 0$, so $\mathbf{B} = \nabla\times\mathbf{A}$). There is a further fact, Helmholtz's theorem, which you will not need to prove but should know: a field that vanishes far away is completely determined by its divergence and its curl. That is why Maxwell's equations have exactly the form they do. Two of them give $\nabla\cdot\mathbf{E}$ and $\nabla\cdot\mathbf{B}$, two give $\nabla\times\mathbf{E}$ and $\nabla\times\mathbf{B}$, and together that is just enough information to fix both fields and no more.

The code verifies both identities symbolically with SymPy, which does the algebra exactly, and then numerically on a grid, where they hold only to the accuracy of the central differences.

**Worked example 20.** $f = x^2y + \sin z$. $\nabla f = (2xy, x^2, \cos z)$. Curl: $x$: $\partial_y\cos z - \partial_z x^2 = 0$; $y$: $\partial_z(2xy) - \partial_x\cos z = 0$; $z$: $\partial_x x^2 - \partial_y(2xy) = 2x - 2x = 0$.


In [ ]:
import sympy as sp
xs, ys, zs = sp.symbols("x y z")
f_sym = xs**2 * ys + sp.sin(zs)
grad = [sp.diff(f_sym, v) for v in (xs, ys, zs)]
def curl(F):
    return [sp.diff(F[2], ys) - sp.diff(F[1], zs), sp.diff(F[0], zs) - sp.diff(F[2], xs), sp.diff(F[1], xs) - sp.diff(F[0], ys)]
def div(F):
    return sp.diff(F[0], xs) + sp.diff(F[1], ys) + sp.diff(F[2], zs)
print("grad f =", grad, "    curl(grad f) =", [sp.simplify(c) for c in curl(grad)])
G = [ys * zs**2, sp.exp(xs) * zs, xs * ys**3]
print("curl G =", curl(G), "    div(curl G) =", sp.simplify(div(curl(G))))

# The same two identities numerically, on the 3D grid from the previous cell.
F3 = X3**2 * Y3 + np.sin(Z3)
g = np.stack(np.gradient(F3, hs, hs, hs), axis=-1)
curl_g = np.stack([np.gradient(g[..., 2], hs, axis=1) - np.gradient(g[..., 1], hs, axis=2),
                   np.gradient(g[..., 0], hs, axis=2) - np.gradient(g[..., 2], hs, axis=0),
                   np.gradient(g[..., 1], hs, axis=0) - np.gradient(g[..., 0], hs, axis=1)], axis=-1)
print(f"numerical |curl(grad f)| max = {np.abs(curl_g[2:-2, 2:-2, 2:-2]).max():.1e}  (typical |grad f| ~ {np.linalg.norm(g, axis=-1).mean():.2f})")

### Exercise: measure a curl yourself

Replace the body of `curl_by_shrinking_loop` so that it estimates $(\nabla\times\mathbf{F})_z$ at a point from the circulation around a small square centred there divided by its area. The check uses $\mathbf{F} = (-y^2, x)$, whose curl is $1 + 2y$.

In [ ]:
def curl_by_shrinking_loop(Fx, Fy, cx, cy, half=1e-3):
    """Estimate the z component of curl F at (cx, cy) from a square loop of half-width half."""
    return circulation_around_square(Fx, Fy, cx, cy, half) / (2 * half) ** 2        # replace with your own version


Fx_t, Fy_t = (lambda x, y: -y**2), (lambda x, y: x)
for cy in (-1.0, 0.0, 0.5, 1.5):
    est = curl_by_shrinking_loop(Fx_t, Fy_t, 0.3, cy)
    print(f"at y = {cy:+.1f}: measured {est:.4f}   exact 1 + 2y = {1 + 2 * cy:.4f}")
    assert abs(est - (1 + 2 * cy)) < 1e-3
print("curl exercise passed")

## 4. Verify

Product rules. Vector calculus has a handful of them, listed in `study/vector_calculus_ref.md`. Two are used constantly in the course:
$$\nabla\cdot(f\mathbf{A}) = f\,\nabla\cdot\mathbf{A} + \mathbf{A}\cdot\nabla f, \qquad \nabla\times(f\mathbf{A}) = f\,\nabla\times\mathbf{A} + \nabla f\times\mathbf{A}.$$
Each is the ordinary product rule applied term by term. The check is symbolic, with SymPy, on a non-trivial $f$ and $\mathbf{A}$, so that the identities hold exactly rather than approximately.

In [ ]:
f_sym = xs * ys**2 + sp.cos(zs)
A = [ys * zs, xs**2, sp.sin(xs * ys)]
fA = [f_sym * a for a in A]
grad_f = [sp.diff(f_sym, v) for v in (xs, ys, zs)]
lhs_div = div(fA); rhs_div = f_sym * div(A) + sum(a * g for a, g in zip(A, grad_f))
lhs_curl = curl(fA); cross = [grad_f[1] * A[2] - grad_f[2] * A[1], grad_f[2] * A[0] - grad_f[0] * A[2], grad_f[0] * A[1] - grad_f[1] * A[0]]
rhs_curl = [f_sym * c + d for c, d in zip(curl(A), cross)]
print("div(fA) - [f div A + A . grad f] =", sp.simplify(lhs_div - rhs_div))
print("curl(fA) - [f curl A + grad f x A] =", [sp.simplify(l - r) for l, r in zip(lhs_curl, rhs_curl)])
assert sp.simplify(lhs_div - rhs_div) == 0 and all(sp.simplify(l - r) == 0 for l, r in zip(lhs_curl, rhs_curl))

## 5. Explore and rebuild

Explore. The field $\mathbf{F} = (-y, x)/(x^2 + y^2)$ is the rotation field divided by $r^2$. Measure its curl with shrinking loops at several points away from the origin. It is zero everywhere you can measure it. Then compute its circulation around the unit circle: $2\pi$. A field with zero curl everywhere should have zero circulation around every loop. Find the loophole (it has to do with where the field is defined), and keep the example in mind: it is the magnetic field of a straight wire, and the puzzle is resolved in notebook 00d and chapter 12.

Rebuild. Close this notebook. On paper, derive the divergence from the flux out of a small box and the $z$ curl from the circulation around a small loop, as in sections 3.3 and 3.4. Then write the two tables of worked examples 4 and 6 from scratch. In a blank editor, write `flux_out_of_square` and `circulation_around_square` and reproduce both tables numerically.

## 6. Key results

All in Cartesian coordinates. Curvilinear forms are in notebook 00d and `study/vector_calculus_ref.md`.

| result | formula | holds when |
|---|---|---|
| forward difference | $(f(x+h) - f(x))/h = f' + O(h)$ | $f$ smooth |
| central difference | $(f(x+h) - f(x-h))/2h = f' + O(h^2)$ | $f$ smooth |
| second difference | $(f(x+h) - 2f(x) + f(x-h))/h^2 = f'' + O(h^2)$ | $f$ smooth |
| mixed partials | $\partial_x\partial_y f = \partial_y\partial_x f$ | $f$ smooth |
| chain rule for $g(r)$ | $\partial_x g(r) = g'(r)\,x/r$ | $r = \sqrt{x^2+y^2+z^2}$ |
| gradient | $\nabla f = (\partial_x f, \partial_y f, \partial_z f)$ | scalar $f$ |
| change along a step | $df = \nabla f\cdot d\mathbf{l}$ | small step |
| directional derivative | $\nabla f\cdot\hat{\mathbf{u}}$, at most $\lvert\nabla f\rvert$ | $\hat{\mathbf{u}}$ a unit vector |
| gradients of $r$ | $\nabla r = \hat{\mathbf{r}}$, $\nabla r^n = nr^{n-1}\hat{\mathbf{r}}$, $\nabla(1/r) = -\hat{\mathbf{r}}/r^2$ | $r > 0$ |
| divergence | $\nabla\cdot\mathbf{F} = \partial_x F_x + \partial_y F_y + \partial_z F_z$ = flux out per unit volume | vector $\mathbf{F}$ |
| Coulomb field | $\nabla\cdot(\mathbf{r}/r^3) = 0$ | $r > 0$ |
| curl | $\nabla\times\mathbf{F} = (\partial_y F_z - \partial_z F_y,\ \partial_z F_x - \partial_x F_z,\ \partial_x F_y - \partial_y F_x)$ = circulation per unit area | vector $\mathbf{F}$ |
| rigid rotation | $\nabla\times(\boldsymbol\omega\times\mathbf{r}) = 2\boldsymbol\omega$ | any constant $\boldsymbol\omega$ |
| Laplacian | $\nabla^2 f = \partial_{xx}f + \partial_{yy}f + \partial_{zz}f = \nabla\cdot(\nabla f)$ | scalar $f$ |
| Laplacian on a grid | $\frac{4}{h^2}(\text{neighbour average} - f_{ij})$ in 2D, $\frac{6}{h^2}(\dots)$ in 3D | uniform grid |
| harmonic radial functions | $\nabla^2(1/r) = 0$ in 3D, $\nabla^2\ln r = 0$ in 2D | $r > 0$ |
| identity 1 | $\nabla\times(\nabla f) = 0$ | always |
| identity 2 | $\nabla\cdot(\nabla\times\mathbf{F}) = 0$ | always |
| converse 1 | $\nabla\times\mathbf{F} = 0 \Rightarrow \mathbf{F} = -\nabla V$ for some $V$ | region without holes |
| converse 2 | $\nabla\cdot\mathbf{F} = 0 \Rightarrow \mathbf{F} = \nabla\times\mathbf{A}$ for some $\mathbf{A}$ | region without holes |
| product rules | $\nabla(fg) = f\nabla g + g\nabla f$; $\nabla\cdot(f\mathbf{A}) = f\nabla\cdot\mathbf{A} + \mathbf{A}\cdot\nabla f$; $\nabla\times(f\mathbf{A}) = f\nabla\times\mathbf{A} + \nabla f\times\mathbf{A}$ | always |
| in the course | $\mathbf{E} = -\nabla V$ (ch. 3); $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ (ch. 4); $\nabla^2V = -\rho/\epsilon_0$ (ch. 6); $\nabla\times\mathbf{B} = \mu_0\mathbf{J}$ (ch. 12); $\nabla\times\mathbf{E} = -\partial_t\mathbf{B}$ (ch. 14) | |

## 7. Warm-up answers

1. The derivative with respect to one variable while the others are held fixed. For $f = x^2y + \sin x$: $\partial f/\partial y = x^2$; the $\sin x$ term is a constant as far as $y$ is concerned.
2. In the direction in which $f$ increases fastest, perpendicular to the contours (level surfaces). Its length is the rate of increase in that direction, the steepest slope.
3. Divergence measures outward flux per unit volume: whether a tiny box around the point is a net source or sink of the field. Curl measures circulation per unit area: whether a tiny paddlewheel at the point would spin, and about which axis. Divergence is the scalar; curl is a vector.
4. Yes to both. The rotation field $(-y, x)$ has divergence $0$ and curl $2$. The radial field $(x, y)$ has divergence $2$ and curl $0$. The two properties are independent.
5. The value at every point equals the average of the values around it (over a small circle, sphere, or the grid neighbours). Such a function has no interior maxima or minima, only saddles.
6. The central difference. Its error is proportional to $h^2$, so halving $h$ cuts the error by four; the forward difference's error is proportional to $h$ and only halves. The first-order error terms cancel in the central difference because it is symmetric about $x$.
7. $2\omega\hat{\mathbf{z}}$: twice the angular velocity, along the rotation axis. The velocity field is $(-\omega y, \omega x, 0)$ and its $z$ curl is $\omega - (-\omega)$.
